# CTDDG Pretraining — PyTorch port

Line-for-line port of the MXNet pretraining notebook. Same architecture, same loss,
same hyperparameters, same paths. Nothing about the *model* has been changed.

**Parameter-count check:** the MXNet version printed `6,986,506`. This port prints
`6,981,514` trainable parameters **+ 4,992** BatchNorm running statistics
(`running_mean`/`running_var`) `= 6,986,506`. MXNet's `collect_params()` counted the
running stats as parameters; PyTorch registers them as buffers. The models are identical.

---
## Known defects carried over unchanged (from `context/01` §5.3)

These are **not** fixed here, because you asked for a faithful port. Fix them before a long run:

1. **`DATA_PATH` is the unfiltered export.** `chembl_toxicity_labels.csv` (1,207,359 rows) still
   contains the BindingDB ligands and the >100-atom molecules. Training on it leaks the
   fine-tuning/test set into pretraining. The filtered set is 1,090,529 molecules.
2. **`SUMMARY_STEP = 5`** writes a checkpoint every 5 iterations (~30 MB each here). For a real
   run set it to 500.
3. **`BATCH_SIZE = 16`** vs the paper's 8 — results will not be comparable to the published tables.

## Deliberate deviations from the MXNet original (all flagged inline with `# PORT:`)

* `optimizer.zero_grad()` added — MXNet Gluon overwrites gradients, PyTorch accumulates them.
  Omitting this is the single most common MXNet→PyTorch porting bug.
* `self.dense(X)` is computed **once** instead of twice in `MoleculeGenerator_RNN._policy`.
  Mathematically identical; the original updated BatchNorm running stats twice per step.
* CSV is read with `usecols` (two columns instead of 34) — the 377 MB file loads in seconds
  instead of minutes, and the `Fingerprint` column never enters RAM.
* Column auto-detection is stricter. The original `next(c for c in df.columns if 'label' in
  c.lower())` would have selected ChEMBL's **`BAO Label`** column, not `Ames_prediction`.
* Per-branch loss logging added (review item **F12**) — logging only, zero effect on training.
* Optional toxic-branch floor (review item **F13**) — **disabled by default** (`TOX_FLOOR = None`),
  so the default objective is exactly the paper's Eq. 8.
* Checkpoints are `ckpt.pt` / `trainer.pt`, not `ckpt.params` / `trainer.status`, so this run
  cannot silently clobber the MXNet checkpoint sitting in the same directory.

# **Configuration and Imports**
## Environment & Imports

In [1]:
import os
import gc
import sys
import glob
import time
import json
import math
import random
import warnings
from abc import ABCMeta, abstractmethod

import numpy as np
import pandas as pd
import networkx as nx
from scipy import sparse

import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger

# Suppress RDKit noisy parse warnings
RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore', category=UserWarning)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Sampler
from torch.autograd.function import once_differentiable

print(f"torch {torch.__version__} | rdkit {rdkit.__version__} | numpy {np.__version__}")

torch 2.4.1+cu121 | rdkit 2026.03.6 | numpy 2.4.6


## Paths, Execution Toggles, and Hyperparameters

In [ ]:
FORCE_RESTART = False

GPU_ID = 0
CTX = torch.device(f"cuda:{GPU_ID}") if torch.cuda.is_available() else torch.device("cpu")
print(f"Active Compute Context: {CTX}")

# PORT: identical defaults to the MXNet notebook; the env override exists only so the same
# file can be driven from a Slurm script without editing it. Unset => exactly the paths below.
DATA_PATH = os.environ.get(
    "CTDDG_DATA_PATH",
    "/home/muhamed/repo/CTDDG/datasets/data/chembl/chembl_with_ames_predictions.csv")
ATOM_TYPES_PATH = os.environ.get(
    "CTDDG_ATOM_TYPES_PATH",
    "/home/muhamed/repo/CTDDG/datasets/data/atom_types.txt")
CHECKPOINT_DIR = os.environ.get(
    "CTDDG_CHECKPOINT_DIR",
    "/home/muhamed/repo/CTDDG/checkpoints")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Standard checkpoint artifact filenames.
# PORT: .pt instead of .params/.status so a torch checkpoint can never be mistaken for the
# MXNet one already present in this directory.
CKPT_PARAMS_FILE = os.path.join(CHECKPOINT_DIR, "ckpt.pt")
CKPT_TRAINER_FILE = os.path.join(CHECKPOINT_DIR, "trainer.pt")
CKPT_CONFIG_FILE = os.path.join(CHECKPOINT_DIR, "configs.json")
LOG_FILE = os.path.join(CHECKPOINT_DIR, "log.out")


# ==============================================================================
# Dataset column selection
# ==============================================================================
# Set explicitly to skip auto-detection. Leave as None to auto-detect (and check the printout).
SMILES_COL = None
TOX_COL = None
LIMIT_MOLECULES = None      # e.g. 50_000 for a smoke test; None = use everything


# ==============================================================================
# Model Architecture & Training Hyperparameters
# ==============================================================================
BATCH_SIZE = 16          # (!) Original paper used 8. See "Known defects" above.
K_PATHS = 5              # K in CTDDG paper (stochastic generation paths per molecule)
P_ALPHA = 0.8            # Randomness parameter alpha in proposal Q_alpha (paper = 0.8)
LAMBDA_TOX = 1.0         # Toxicity trade-off parameter lambda in CTDDG Eq. 8

# Graph Neural Network Dimensions
F_E = 16                 # Atom embedding size
F_H = [32, 64, 128, 128, 256, 256]  # Output sizes of the 6 GCN layers
F_SKIP = 256             # Output size of skip-connection layer
F_C = [512]              # Hidden sizes of post-GCN fully connected layers
FH_POLICY = 128          # Hidden size for the policy head
ACTIVATION = 'relu'      # GCN activation function
N_RNN = 3                # Number of GRU layers
D_RECEPTIVE = 2          # Graph receptive distance (adds D_2 and D_3 neighborhoods)
MAX_ITER = 120           # Cap on generation/action steps per molecule

# Optimizer & Scheduling
LR = 1e-3                # Initial learning rate
DECAY = 1e-3             # LEARNING-RATE decay factor, NOT weight decay.
DECAY_STEP = 100         # lr <- lr * (1 - DECAY) every DECAY_STEP iterations
CLIP_GRAD = 3.0          # Element-wise gradient clip (MXNet optimizer `clip_gradient` semantics)
SUMMARY_STEP = 500         # (!) Debug value. Set to 500 for a real run.
TOTAL_ITERATIONS = 480000

# Runtime
NUM_WORKERS = 8
SEED = 42
ALLOW_TF32 = True        # Big speedup on Ada/Ampere (L40, A30); negligible precision impact.
TOX_FLOOR = None         # Review item F13. None = paper-exact Eq. 8. Try -150.0 if the toxic
                         # branch diverges (loss marching to large negatives).

Active Compute Context: cuda:0


## Verification & Sanity Check

In [3]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = ALLOW_TF32
    torch.backends.cudnn.allow_tf32 = ALLOW_TF32

assert os.path.exists(DATA_PATH), f"Data file not found at: {DATA_PATH}"
assert os.path.exists(ATOM_TYPES_PATH), f"Atom types file not found at: {ATOM_TYPES_PATH}"

# Test device tensor allocation
_test_tensor = torch.zeros((1,), device=CTX)
_ = _test_tensor.cpu().numpy()

print("[OK] Paths verified.")
print(f"[OK] Torch device {CTX} is fully responsive.")
if CTX.type == "cuda":
    print(f"[OK] GPU: {torch.cuda.get_device_name(GPU_ID)}")
print(f"FORCE_RESTART is set to: {FORCE_RESTART}")

[OK] Paths verified.
[OK] Torch device cuda:0 is fully responsive.
[OK] GPU: NVIDIA L40
FORCE_RESTART is set to: False


# **Data Loading & Batching**
## Molecular Specification & Graph Traversal Utilities

This whole section is pure NumPy / NetworkX / RDKit and is **byte-identical** to the MXNet
version — there was no framework code in it to port.

In [4]:
class MoleculeSpec(object):
    """Encodes and decodes atom/bond types using the immutable atom vocabulary."""
    def __init__(self, file_name=ATOM_TYPES_PATH):
        self.atom_types = []
        self.atom_symbols = []
        with open(file_name, 'r') as f:
            for line in f:
                atom_type_i = line.strip('\n').split(',')
                self.atom_types.append((atom_type_i[0], int(atom_type_i[1]), int(atom_type_i[2])))
                if atom_type_i[0] not in self.atom_symbols:
                    self.atom_symbols.append(atom_type_i[0])

        self.bond_orders = [
            Chem.BondType.AROMATIC,
            Chem.BondType.SINGLE,
            Chem.BondType.DOUBLE,
            Chem.BondType.TRIPLE
        ]
        self.max_iter = MAX_ITER

    def get_atom_type(self, atom):
        return self.atom_types.index((atom.GetSymbol(), atom.GetFormalCharge(), atom.GetNumExplicitHs()))

    def get_bond_type(self, bond):
        return self.bond_orders.index(bond.GetBondType())

    def index_to_atom(self, idx):
        symbol, charge, hs = self.atom_types[idx]
        a = Chem.Atom(symbol)
        a.SetFormalCharge(charge)
        a.SetNumExplicitHs(hs)
        return a

    def index_to_bond(self, mol, begin_id, end_id, idx):
        mol.AddBond(begin_id, end_id, self.bond_orders[idx])

    @property
    def num_atom_types(self):
        return len(self.atom_types)

    @property
    def num_bond_types(self):
        return len(self.bond_orders)


_mol_spec = None


def get_mol_spec():
    global _mol_spec
    if _mol_spec is None:
        _mol_spec = MoleculeSpec(ATOM_TYPES_PATH)
    return _mol_spec


def get_graph_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES string: {smiles}")

    atom_types, atom_ranks, bonds, bond_types = [], [], [], []
    for a, r in zip(mol.GetAtoms(), Chem.CanonicalRankAtoms(mol)):
        atom_types.append(get_mol_spec().get_atom_type(a))
        atom_ranks.append(r)

    for b in mol.GetBonds():
        idx_1, idx_2 = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bt = get_mol_spec().get_bond_type(b)
        bonds.append([idx_1, idx_2])
        bond_types.append(bt)

    graph = nx.Graph()
    graph.add_nodes_from(range(len(atom_types)))
    graph.add_edges_from(bonds)
    return graph, atom_types, atom_ranks, bonds, bond_types


def traverse_graph(graph, atom_ranks, current_node=None, step_ids=None, p=P_ALPHA, log_p=0.0):
    """Importance-sampling proposal Q_alpha traversal using canonical rank."""
    if current_node is None:
        next_nodes = range(len(atom_ranks))
        step_ids = [-1] * len(next_nodes)
        next_node_ranks = atom_ranks
    else:
        next_nodes = list(graph.neighbors(current_node))
        next_nodes = [n for n in next_nodes if step_ids[n] < 0]
        next_node_ranks = [atom_ranks[n] for n in next_nodes]

    next_nodes = [n for n, r in sorted(zip(next_nodes, next_node_ranks), key=lambda x: x[1])]

    while len(next_nodes) > 0:
        if len(next_nodes) == 1:
            next_node = next_nodes[0]
        elif random.random() >= (1.0 - p):
            next_node = next_nodes[0]
            log_p += np.log(p)
        else:
            next_node = next_nodes[random.randint(1, len(next_nodes) - 1)]
            log_p += np.log((1.0 - p) / (len(next_nodes) - 1))

        step_ids[next_node] = max(step_ids) + 1
        _, log_p = traverse_graph(graph, atom_ranks, next_node, step_ids, p, log_p)
        next_nodes = [n for n in graph.neighbors(next_node) if step_ids[n] < 0]
        next_node_ranks = [atom_ranks[n] for n in next_nodes]
        next_nodes = [n for n, r in sorted(zip(next_nodes, next_node_ranks), key=lambda x: x[1])]

    return step_ids, log_p


def single_reorder(X_0, A_0, step_ids):
    X_0, A_0 = np.copy(X_0), np.copy(A_0)
    step_ids = np.array(step_ids, dtype=np.int32)
    sorted_ids = np.argsort(step_ids)
    X_0 = X_0[sorted_ids]
    A_0[:, 0], A_0[:, 1] = step_ids[A_0[:, 0]], step_ids[A_0[:, 1]]

    max_b, min_b = np.amax(A_0[:, :2], axis=1), np.amin(A_0[:, :2], axis=1)
    A_0 = A_0[np.lexsort([-min_b, max_b]), :]

    max_b, min_b = np.amax(A_0[:, :2], axis=1), np.amin(A_0[:, :2], axis=1)
    is_append = np.concatenate([np.array([True]), max_b[1:] > max_b[:-1]])
    A_0 = np.concatenate([
        np.where(is_append[:, np.newaxis], np.stack([min_b, max_b], axis=1), np.stack([max_b, min_b], axis=1)),
        A_0[:, -1:]
    ], axis=1)
    return X_0, A_0


def single_expand(X_0, A_0):
    X_0, A_0 = np.copy(X_0), np.copy(A_0)
    is_append_iter = np.less(A_0[:, 0], A_0[:, 1]).astype(np.int32)

    NX = np.cumsum(np.pad(is_append_iter, [[1, 0]], mode='constant', constant_values=1))
    shift = np.cumsum(np.pad(NX, [[1, 0]], mode='constant')[:-1])
    X_index = np.arange(NX.sum(), dtype=np.int32) - np.repeat(shift, NX)
    X = X_0[X_index]

    _, A_index = np.tril_indices(A_0.shape[0])
    A = A_0[A_index, :]
    NA = np.arange(A_0.shape[0] + 1)

    action_type = 1 - is_append_iter
    atom_type = np.where(action_type == 0, X_0[A_0[:, 1]], 0)
    bond_type = A_0[:, 2]
    append_pos = np.where(action_type == 0, A_0[:, 0], 0)
    connect_pos = np.where(action_type == 1, A_0[:, 1], 0)
    actions = np.stack([action_type, atom_type, bond_type, append_pos, connect_pos], axis=1)
    actions = np.append(actions, [[2, 0, 0, 0, 0]], axis=0)
    action_0 = np.array([X_0[0]], dtype=np.int32)

    last_atom_index = shift + NX - 1
    last_atom_mask = np.zeros_like(X)
    last_atom_mask[last_atom_index] = np.where(
        np.pad(is_append_iter, [[1, 0]], mode='constant', constant_values=1) == 1,
        np.ones_like(last_atom_index),
        np.ones_like(last_atom_index) * 2
    )
    return action_0, X, NX, A, NA, actions, last_atom_mask


def get_d(A, X):
    def _to_sparse(_A, _X):
        return sparse.coo_matrix(
            (np.ones([_A.shape[0] * 2], dtype=np.int32),
             (np.concatenate([_A[:, 0], _A[:, 1]], axis=0),
              np.concatenate([_A[:, 1], _A[:, 0]], axis=0))),
            shape=[_X.shape[0]] * 2
        )

    A_sparse = _to_sparse(A, X)
    d2 = A_sparse * A_sparse
    d3 = d2 * A_sparse

    D_2 = np.stack(d2.nonzero(), axis=1)
    D_2 = D_2[D_2[:, 0] < D_2[:, 1], :]

    D_3 = np.stack(d3.nonzero(), axis=1)
    D_3 = D_3[D_3[:, 0] < D_3[:, 1], :]

    D_3_sparse = _to_sparse(D_3, X)
    D_3_sparse = D_3_sparse - D_3_sparse.multiply(A_sparse)
    D_3 = np.stack(D_3_sparse.nonzero(), axis=1)
    D_3 = D_3[D_3[:, 0] < D_3[:, 1], :]
    return D_2, D_3


def merge_single_0(X_0, A_0, NX_0, NA_0):
    cumsum = np.cumsum(np.pad(NX_0, [[1, 0]], mode='constant')[:-1])
    A_0[:, :2] += np.stack([np.repeat(cumsum, NA_0)] * 2, axis=1)
    D_0_2, D_0_3 = get_d(A_0, X_0)

    A_split = []
    for i in range(get_mol_spec().num_bond_types):
        A_split.append(A_0[A_0[:, 2] == i, :2])
    A_split.extend([D_0_2, D_0_3])
    A_0 = A_split
    NX_rep_0 = np.repeat(np.arange(NX_0.shape[0]), NX_0)
    return X_0, A_0, NX_0, NX_rep_0


def merge_single(X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions, last_append_mask, log_p):
    X, A, NX, NX_rep = merge_single_0(X, A, NX, NA)
    cumsum = np.cumsum(np.pad(NX, [[1, 0]], mode='constant')[:-1])
    actions[:, -2] += cumsum * (actions[:, 0] == 0)
    actions[:, -1] += cumsum * (actions[:, 0] == 1)
    mol_ids_rep = np.repeat(mol_ids, NX)
    rep_ids_rep = np.repeat(rep_ids, NX)
    return (X, A, mol_ids_rep, rep_ids_rep, iw_ids, last_append_mask, NX, NX_rep, action_0, actions, log_p)

## Data Ingestion & Sampler

In [5]:
# PORT: stricter than the original one-liner. ChEMBL's export contains a column literally named
# "BAO Label", which `'label' in c.lower()` matched *before* reaching `Ames_prediction`.
_PREFERRED_SMILES = ["Smiles", "smiles", "SMILES", "canonical_smiles",
                     "Canonical_Smiles", "CanonicalSmiles"]
_PREFERRED_TOX = ["Ames_prediction", "Ames_Prediction", "ames_prediction",
                  "Toxic_class", "tox_class", "Toxicity", "toxicity"]


def _resolve_columns(columns):
    """Pick the SMILES and toxicity-label columns, preferring exact known names."""
    cols = list(columns)
    lower = {c.lower(): c for c in cols}

    smi = SMILES_COL
    if smi is None:
        for cand in _PREFERRED_SMILES:
            if cand in cols:
                smi = cand
                break
    if smi is None:
        smi = next((c for c in cols if 'smiles' in c.lower()), cols[0])

    tox = TOX_COL
    if tox is None:
        for cand in _PREFERRED_TOX:
            if cand in cols:
                tox = cand
                break
    if tox is None:
        # Narrow substring fallback: 'ames'/'toxic' only, never bare 'label'/'class'.
        tox = next((c for c in cols
                    if ('ames' in c.lower() or 'toxic' in c.lower())
                    and c != smi), None)
    if tox is None:
        raise ValueError(
            f"Could not identify a toxicity-label column among {cols[:40]}. "
            f"Set TOX_COL explicitly."
        )
    if smi not in cols or tox not in cols:
        raise ValueError(f"Resolved columns not present. smiles={smi!r} tox={tox!r}")
    return smi, tox


def load_chembl_dataset(file_path):
    """Loads SMILES and binary Ames labels from CSV or space-separated files."""
    try:
        header = pd.read_csv(file_path, nrows=0)
        smiles_col, tox_col = _resolve_columns(header.columns)
        print(f"[INFO] Using SMILES column {smiles_col!r} and toxicity column {tox_col!r}")

        # PORT: usecols keeps the 377 MB / 34-column file (incl. the huge `Fingerprint`
        # column) out of RAM. Semantically identical to reading the whole frame.
        df = pd.read_csv(file_path, usecols=[smiles_col, tox_col])

        n_before = len(df)
        df = df.dropna(subset=[smiles_col, tox_col])
        if len(df) != n_before:
            print(f"[INFO] Dropped {n_before - len(df)} rows with missing SMILES or label.")

        smiles_list = df[smiles_col].astype(str).tolist()
        tox_list = df[tox_col].astype(float).astype(int).tolist()
    except Exception as e:
        print(f"[INFO] CSV path failed ({type(e).__name__}: {e}); falling back to "
              f"plain delimited parsing.")
        smiles_list, tox_list = [], []
        with open(file_path, 'r') as f:
            for line in f:
                parts = line.strip().replace(',', ' ').split()
                if len(parts) >= 2:
                    smiles_list.append(parts[0])
                    tox_list.append(int(float(parts[1])))
                elif len(parts) == 1:
                    smiles_list.append(parts[0])
                    tox_list.append(0)

    records = list(zip(smiles_list, tox_list))
    if LIMIT_MOLECULES is not None:
        records = records[:LIMIT_MOLECULES]
        print(f"[WARN] LIMIT_MOLECULES is set: using only {len(records)} molecules.")
    return records


class BalancedSampler(Sampler):
    """Balances mini-batch lengths to minimize padding overhead during GCN/RNN computation."""
    def __init__(self, cost, batch_size):
        index = np.argsort(cost).tolist()
        chunk_size = int(float(len(cost)) / batch_size)
        self.index = []
        for i in range(batch_size):
            self.index.append(index[i * chunk_size:(i + 1) * chunk_size])

    def _g(self):
        for index_i in self.index:
            random.shuffle(index_i)
        for batch_index in zip(*self.index):
            yield list(batch_index)

    def __iter__(self):
        return self._g()

    def __len__(self):
        return len(self.index[0])

## Graph & RNN Data Loaders (With OOV Shield)

In [6]:
def process_single(record, k=K_PATHS, p=P_ALPHA):
    """Extracts graph, computes K proposal paths, and binds true Ames labels."""
    if isinstance(record, (tuple, list)):
        smiles, tox_label = record[0], int(record[1])
    elif " " in record:
        parts = record.split(" ")
        smiles, tox_label = parts[0], int(parts[1])
    else:
        smiles, tox_label = record, 0

    graph, atom_types, atom_ranks, bonds, bond_types = get_graph_from_smiles(smiles)
    X_0 = np.array(atom_types, dtype=np.int32)
    A_0 = np.concatenate([
        np.array(bonds, dtype=np.int32),
        np.array(bond_types, dtype=np.int32)[:, np.newaxis]
    ], axis=1) if len(bonds) > 0 else np.zeros((0, 3), dtype=np.int32)

    X, A, NX, NA = [], [], [], []
    mol_ids, rep_ids, iw_ids = [], [], []
    action_0, actions, last_append_mask = [], [], []
    log_p = []

    tox_class = np.full((k,), tox_label, dtype=np.int32)

    for i in range(k):
        step_ids_i, log_p_i = traverse_graph(graph, atom_ranks, p=p)
        X_i, A_i = single_reorder(X_0, A_0, step_ids_i)
        action_0_i, X_i, NX_i, A_i, NA_i, actions_i, last_atom_mask_i = single_expand(X_i, A_i)

        X.append(X_i)
        A.append(A_i)
        NX.append(NX_i)
        NA.append(NA_i)
        action_0.append(action_0_i)
        actions.append(actions_i)
        last_append_mask.append(last_atom_mask_i)
        mol_ids.append(np.zeros_like(NX_i, dtype=np.int32))
        rep_ids.append(np.ones_like(NX_i, dtype=np.int32) * i)
        iw_ids.append(np.ones_like(NX_i, dtype=np.int32) * i)
        log_p.append(log_p_i)

    return (
        np.concatenate(X, axis=0), np.concatenate(A, axis=0),
        np.concatenate(NX, axis=0), np.concatenate(NA, axis=0),
        np.concatenate(mol_ids, axis=0), np.concatenate(rep_ids, axis=0),
        np.concatenate(iw_ids, axis=0), np.concatenate(action_0, axis=0),
        np.concatenate(actions, axis=0), np.concatenate(last_append_mask, axis=0),
        np.array(log_p, dtype=np.float32), tox_class
    )


class MolRNNCollate:
    """Standalone collator; picklable so it survives DataLoader worker start-up."""
    def __init__(self, k=K_PATHS, p=P_ALPHA):
        self.k = k
        self.p = p

    def __call__(self, batch):
        # 1. SHIELD: Pre-filter out any molecules with Out-Of-Vocabulary atoms
        valid_batch = []
        for record in batch:
            smiles = record[0] if isinstance(record, (tuple, list)) else (record.split()[0] if " " in record else record)
            try:
                _ = get_graph_from_smiles(smiles)
                valid_batch.append(record)
            except ValueError:
                continue  # Safely skip OOV atoms

        if not valid_batch:
            return None

        # 2. Process only the valid, guaranteed-to-parse records
        shapes = [[0], [0, 3], [0], [0], [0], [0], [0], [0], [0, 5], [0], [0], [0]]
        dtypes = [np.int32, np.int32, np.int32, np.int32, np.int32, np.int32,
                  np.int32, np.int32, np.int32, np.int32, np.float32, np.int32]

        _build = lambda: [np.zeros(shape=s, dtype=d) for s, d in zip(shapes, dtypes)]
        _append = lambda _r0, _r1: [np.concatenate([__r0, __r1], axis=0) for __r0, __r1 in zip(_r0, _r1)]

        X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions, last_append_mask, log_p, tox_class = _build()

        valid_count = 0
        for record in valid_batch:
            (X_i, A_i, NX_i, NA_i, mol_ids_i, rep_ids_i, iw_ids_i,
             action_0_i, actions_i, last_append_mask_i, log_p_i, tox_class_i) = process_single(record, self.k, self.p)

            if valid_count != 0:
                mol_ids_i += mol_ids[-1] + 1
                iw_ids_i += iw_ids[-1] + 1

            (X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions, last_append_mask, log_p, tox_class) = _append(
                [X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions, last_append_mask, log_p, tox_class],
                [X_i, A_i, NX_i, NA_i, mol_ids_i, rep_ids_i, iw_ids_i, action_0_i, actions_i,
                 last_append_mask_i, log_p_i, tox_class_i.reshape(-1)]
            )
            valid_count += 1

        X, A, mol_ids_rep, rep_ids_rep, iw_ids, last_append_mask, NX, NX_rep, action_0, actions, log_p = merge_single(
            X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions, last_append_mask, log_p
        )
        result_out = [X, A, mol_ids_rep, rep_ids_rep, iw_ids, last_append_mask, NX, NX_rep,
                      action_0, actions, log_p, tox_class]

        # RNN mapping logic
        smiles_batch = [b[0] if isinstance(b, (tuple, list)) else b.split()[0] for b in valid_batch]
        mol_list = [Chem.MolFromSmiles(s) for s in smiles_batch]

        graph_to_rnn = np.zeros((len(valid_batch), self.k, get_mol_spec().max_iter), dtype=np.int32)
        rnn_to_graph = []
        cum_sum = 0

        for i, mol_i in enumerate(mol_list):
            num_iter = mol_i.GetNumBonds() + 1
            for k_idx in range(self.k):
                graph_to_rnn[i, k_idx, :num_iter] = np.arange(num_iter) + cum_sum
                rnn_to_graph_0 = np.ones([num_iter], dtype=np.int32) * i
                rnn_to_graph_1 = np.ones_like(rnn_to_graph_0) * k_idx
                rnn_to_graph_2 = np.arange(num_iter)
                rnn_to_graph.append(np.stack([rnn_to_graph_0, rnn_to_graph_1, rnn_to_graph_2], axis=0))
                cum_sum += num_iter

        rnn_to_graph = np.concatenate(rnn_to_graph, axis=1)
        NX_cum = np.cumsum(result_out[6])
        return result_out + [graph_to_rnn, rnn_to_graph, NX_cum]


def batch_to_tensor(record, device=None):
    """Move one collated batch onto the compute device.

    PORT: MXNet `nd.sparse.csr_matrix` -> `torch.sparse_coo_tensor(...).coalesce()`.
    All index tensors become int64, because PyTorch indexing/embedding/index_add require it.
    """
    device = CTX if device is None else device
    *base_records, graph_to_rnn, rnn_to_graph, NX_cum = record
    (X, A, mol_ids_rep, rep_ids_rep, iw_ids, last_append_mask,
     NX, NX_rep, action_0, actions, log_p, tox_class) = base_records

    n_nodes = int(X.shape[0])
    X = torch.as_tensor(X, dtype=torch.long, device=device)

    A_sparse = []
    for A_i in A:
        if A_i.shape[0] == 0:
            A_sparse.append(None)
        else:
            A_i = np.concatenate([A_i, A_i[:, [1, 0]]], axis=0)   # symmetrise
            idx = torch.as_tensor(A_i[:, :2].T.astype(np.int64), device=device)
            val = torch.ones(A_i.shape[0], dtype=torch.float32, device=device)
            A_sparse.append(
                torch.sparse_coo_tensor(idx, val, (n_nodes, n_nodes),
                                        dtype=torch.float32, device=device).coalesce()
            )

    batch_size = int(mol_ids_rep.max() + 1)
    iw_size = int(rep_ids_rep.max() + 1)

    iw_ids, last_append_mask, NX, NX_rep, action_0, actions = [
        torch.as_tensor(_x, dtype=torch.long, device=device)
        for _x in [iw_ids, last_append_mask, NX, NX_rep, action_0, actions]
    ]
    log_p = torch.as_tensor(log_p, dtype=torch.float32, device=device)
    tox_class = torch.as_tensor(tox_class, dtype=torch.float32, device=device)

    graph_to_rnn = torch.as_tensor(graph_to_rnn, dtype=torch.long, device=device)
    rnn_to_graph = torch.as_tensor(rnn_to_graph, dtype=torch.long, device=device)
    NX_cum = torch.as_tensor(NX_cum, dtype=torch.long, device=device)

    return [X, A_sparse, iw_ids, last_append_mask, NX, NX_rep, action_0, actions, log_p,
            batch_size, iw_size, tox_class, graph_to_rnn, rnn_to_graph, NX_cum]

## Data Verification & Mini-Batch Smoke Test

In [7]:
# 1. Ingest dataset
dataset = load_chembl_dataset(DATA_PATH)
print(f"Total molecules loaded: {len(dataset)}")

labels = [rec[1] for rec in dataset]
num_toxic = sum(labels)
num_nontoxic = len(labels) - num_toxic
print(f"Class distribution: Non-Toxic (0) = {num_nontoxic}, Toxic (1) = {num_toxic}")

# 2. Build sampler and data loader
sampler_train = BalancedSampler(cost=[len(rec[0]) for rec in dataset], batch_size=BATCH_SIZE)
collate_fn = MolRNNCollate(k=K_PATHS, p=P_ALPHA)
loader_train = DataLoader(
    dataset,
    batch_sampler=sampler_train,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,                      # PORT: MXNet `batchify_fn` -> `collate_fn`
    persistent_workers=(NUM_WORKERS > 0),
)

# 3. Pull a single batch and move it to the device
batch_cpu = next(iter(loader_train))
batch_gpu = batch_to_tensor(batch_cpu)

(X, A_sparse, iw_ids, last_append_mask, NX, NX_rep, action_0, actions,
 log_p, batch_size, iw_size, tox_class, graph_to_rnn, rnn_to_graph, NX_cum) = batch_gpu

print("\n--- Mini-Batch Shape & Sanity Check ---")
print(f"Batch size (N): {batch_size}, Paths per molecule (K): {iw_size}")
print(f"Node features X shape: {tuple(X.shape)}, Device: {X.device}")
print(f"Adjacency blocks: {len(A_sparse)} (= N_B + D = 4 + 2)")
print("Verification passed!!!")

[INFO] Using SMILES column 'Smiles' and toxicity column 'Ames_prediction'
Total molecules loaded: 1090529
Class distribution: Non-Toxic (0) = 830371, Toxic (1) = 260158

--- Mini-Batch Shape & Sanity Check ---
Batch size (N): 16, Paths per molecule (K): 5
Node features X shape: (29531,), Device: cuda:0
Adjacency blocks: 6 (= N_B + D = 4 + 2)
Verification passed!!!


# **Model Classes, Eq. 8 Loss, & Checkpoint Loading Logic**
## Torch Graph Math & Primitives

In [8]:
def segment_sum(X, idx, num_seg):
    """out[idx[i]] += X[i].  Replaces MXNet's SegmentSumFn sparse-matmul trick.

    Handles trailing dims of any rank, matching SegmentSumFn's reshape-then-dot behaviour.
    Natively differentiable: the gradient is a gather, which is exactly what multiplying by
    the transposed segment matrix did in the MXNet version.
    """
    out = torch.zeros((num_seg,) + tuple(X.shape[1:]), dtype=X.dtype, device=X.device)
    return out.index_add(0, idx, X)


def _concat_neighbourhoods(X, A_list):
    """[X, A_1 X, ..., A_4 X, D_2 X, D_3 X] -> [n, F_in * (|B| + D + 1)]"""
    parts = [X]
    for A in A_list:
        parts.append(torch.sparse.mm(A, X) if A is not None else torch.zeros_like(X))
    return torch.cat(parts, dim=1)


class EfficientGraphConvFn(torch.autograd.Function):
    """Saves memory by re-computing the concatenated neighbourhood in backward.

    Every A in A_list is symmetric (built by symmetrising the edge list in
    `batch_to_tensor`), so A^T == A and backward can reuse A directly.
    """

    @staticmethod
    def forward(ctx, X, W, A_list):
        ctx.A_list = A_list
        ctx.save_for_backward(X, W)
        X_out = _concat_neighbourhoods(X, A_list)
        return X_out @ W

    @staticmethod
    @once_differentiable
    def backward(ctx, grad_output):
        X, W = ctx.saved_tensors
        A_list = ctx.A_list

        X_out = _concat_neighbourhoods(X, A_list)          # recomputed, not stored
        grad_output = grad_output.contiguous()

        grad_W = X_out.t() @ grad_output
        grad_X_out = grad_output @ W.t()

        chunks = torch.split(grad_X_out, X.shape[1], dim=1)
        grad_X = chunks[0]
        for A, g in zip(A_list, chunks[1:]):
            if A is not None:
                grad_X = grad_X + torch.sparse.mm(A, g.contiguous())
        return grad_X, grad_W, None


def get_activation(name):
    return {'relu': torch.relu, 'tanh': torch.tanh}[name]

## Neural Network Layers

In [ ]:
class Linear_BN(nn.Sequential):
    """Dense(no bias) -> BatchNorm.  MXNet BatchNorm(momentum=0.9) == torch(momentum=0.1)."""
    def __init__(self, F_in, F_out):
        super(Linear_BN, self).__init__(
            nn.Linear(F_in, F_out, bias=False),
            nn.BatchNorm1d(F_out, eps=1e-5, momentum=0.1),
        )


class GraphConv(nn.Module):
    def __init__(self, Fin, Fout, D):
        super(GraphConv, self).__init__()
        self.Fin = Fin
        self.Fout = Fout
        self.D = D                       # = N_B + D_receptive = number of adjacency blocks
        self.W = nn.Parameter(torch.empty(self.Fin * (self.D + 1), self.Fout))

    def forward(self, X, A_list):
        return EfficientGraphConvFn.apply(X, self.W, A_list)


class Policy(nn.Module):
    def __init__(self, F_in, F_h, N_A, N_B, k=1):
        super(Policy, self).__init__()
        self.F_in, self.F_h, self.N_A, self.N_B, self.k = F_in, F_h, N_A, N_B, k
        self.linear_h = Linear_BN(F_in * 2, self.F_h * k)
        self.linear_h_t = Linear_BN(F_in, self.F_h * k)
        self.linear_x = nn.Linear(self.F_h, self.N_B + self.N_B * self.N_A)
        self.linear_x_t = nn.Linear(self.F_h, 1)
        self.linear_pi = nn.Linear(self.F_in, self.k) if self.k > 1 else None

    def forward(self, X, NX, NX_rep, X_end=None):
        if X_end is None:
            X_end = segment_sum(X, NX_rep, NX.shape[0]) / NX.unsqueeze(1).float()
        X = torch.cat([X, X_end[NX_rep, :]], dim=1)

        X_h = torch.relu(self.linear_h(X)).reshape(-1, self.F_h)
        X_h_end = torch.relu(self.linear_h_t(X_end)).reshape(-1, self.F_h)

        # Clip logits to +/-80 before exp() to stay inside float32 range (as in the original).
        raw_logits_x = self.linear_x(X_h).reshape(-1, self.k, self.N_B + self.N_B * self.N_A)
        raw_logits_x_t = self.linear_x_t(X_h_end).reshape(-1, self.k, 1)

        safe_logits_x = torch.clamp(raw_logits_x, -80.0, 80.0)
        safe_logits_x_t = torch.clamp(raw_logits_x_t, -80.0, 80.0)

        X_x = torch.exp(safe_logits_x)
        X_x_end = torch.exp(safe_logits_x_t)

        # Joint normalisation over every action of every atom, plus `end`, per molecule.
        X_sum = segment_sum(X_x, NX_rep, NX.shape[0]).sum(-1, keepdim=True) + X_x_end
        X_sum_gathered = X_sum[NX_rep, :, :]

        X_softmax = X_x / (X_sum_gathered + 1e-10)
        X_softmax_end = X_x_end / (X_sum + 1e-10)

        if self.k > 1:
            pi = torch.softmax(self.linear_pi(X_end), dim=1).unsqueeze(-1)
            pi_gathered = pi[NX_rep, :, :]
            X_softmax = (X_softmax * pi_gathered).sum(dim=1)
            X_softmax_end = (X_softmax_end * pi).sum(dim=1)
        else:
            X_softmax = X_softmax.squeeze(1)
            X_softmax_end = X_softmax_end.squeeze(1)

        connect, append = X_softmax[:, :self.N_B], X_softmax[:, self.N_B:]
        append = append.reshape(-1, self.N_A, self.N_B)
        end = X_softmax_end.squeeze(-1)
        return append, connect, end

## Generator & Eq. 8 Toxicity-Weighted Loss

In [10]:
class MoleculeGenerator(nn.Module):
    __metaclass__ = ABCMeta

    def __init__(self, N_A, N_B, D, F_e, F_skip, F_c, Fh_policy, activation, *args, **kwargs):
        super(MoleculeGenerator, self).__init__()
        self.N_A, self.N_B, self.D = N_A, N_B, D
        self.F_e, self.F_skip, self.Fh_policy = F_e, F_skip, Fh_policy
        self.F_c = list(F_c) if isinstance(F_c, tuple) else list(F_c)
        self.activation = get_activation(activation)

        self.embedding_atom = nn.Embedding(self.N_A, self.F_e)
        self.embedding_mask = nn.Embedding(3, self.F_e)
        self._build_graph_conv(*args, **kwargs)

        self.dense = nn.Sequential()
        for i, (f_in, f_out) in enumerate(zip([self.F_skip] + self.F_c[:-1], self.F_c)):
            self.dense.add_module(str(i), Linear_BN(f_in, f_out))

        self.policy_0 = nn.Parameter(torch.zeros(self.N_A))
        self.policy_h = Policy(self.F_c[-1], self.Fh_policy, self.N_A, self.N_B)
        self.mode = 'loss'

        # F12 bookkeeping: filled in every forward, read by the training loop. Detached.
        self.last_l = None
        self.last_tox = None

    @abstractmethod
    def _build_graph_conv(self, *args, **kwargs): pass

    @abstractmethod
    def _graph_conv_forward(self, X, A): pass

    def _policy_0(self):
        return torch.softmax(self.policy_0, dim=0)

    def _policy(self, X, A, NX, NX_rep, last_append_mask):
        X = self.embedding_atom(X) + self.embedding_mask(last_append_mask)
        X = self._graph_conv_forward(X, A)
        X = self.dense(X)
        return self.policy_h(X, NX, NX_rep)

    def _likelihood(self, init, append, connect, end, action_0, actions, iw_ids,
                    log_p_sigma, batch_size, iw_size, tox_class_batch):
        action_type, node_type, edge_type, append_pos, connect_pos = (
            actions[:, 0], actions[:, 1], actions[:, 2], actions[:, 3], actions[:, 4])

        def _log_mask(_x, _mask):
            return _mask * torch.log(_x + 1e-10) + (1 - _mask) * torch.zeros_like(_x)

        init = init.reshape(batch_size * iw_size, self.N_A)
        rows = torch.arange(action_0.shape[0], device=action_0.device)
        loss_init = torch.log(init[rows, action_0] + 1e-10)

        loss_end = _log_mask(end, (action_type == 2).to(end.dtype))
        loss_append = _log_mask(append[append_pos, node_type, edge_type],
                                (action_type == 0).to(end.dtype))
        loss_connect = _log_mask(connect[connect_pos, edge_type],
                                 (action_type == 1).to(end.dtype))

        log_p_x = loss_end + loss_append + loss_connect
        log_p_x = segment_sum(log_p_x.unsqueeze(-1), iw_ids, batch_size * iw_size).squeeze(-1)
        log_p_x = log_p_x + loss_init

        log_p_x = log_p_x.reshape(batch_size, iw_size)
        log_p_sigma = log_p_sigma.reshape(batch_size, iw_size)

        # --- Original CDGCN log-likelihood (Eq. 3) -----------------------------------
        l = log_p_x - log_p_sigma
        l = torch.logsumexp(l, dim=1) - math.log(float(iw_size))

        # --- CTDDG Eq. 8 -------------------------------------------------------------
        # All K paths of a molecule carry the same label, so column 0 is the label.
        tox_labels = tox_class_batch.reshape(batch_size, iw_size)[:, 0]

        if TOX_FLOOR is not None:                      # review item F13 (off by default)
            l = torch.where(tox_labels > 0, torch.clamp(l, min=TOX_FLOOR), l)

        tox_weight = 1.0 - (1.0 + LAMBDA_TOX) * tox_labels
        l_weighted = l * tox_weight

        self.last_l = l.detach()
        self.last_tox = tox_labels.detach()
        return l_weighted

    def forward(self, X, A, iw_ids, last_append_mask, NX, NX_rep, action_0, actions,
                log_p, batch_size, iw_size, tox_class, *rnn_args):
        init = self._policy_0().unsqueeze(0).expand(batch_size * iw_size, -1)
        append, connect, end = self._policy(X, A, NX, NX_rep, last_append_mask, *rnn_args)
        l_weighted = self._likelihood(init, append, connect, end, action_0, actions,
                                      iw_ids, log_p, batch_size, iw_size, tox_class)
        if self.mode == 'likelihood':
            return l_weighted
        return -l_weighted.mean()


class MoleculeGenerator_RNN(MoleculeGenerator):
    __metaclass__ = ABCMeta

    def __init__(self, N_A, N_B, D, F_e, F_skip, F_c, Fh_policy, activation, N_rnn, *args, **kwargs):
        super(MoleculeGenerator_RNN, self).__init__(
            N_A, N_B, D, F_e, F_skip, F_c, Fh_policy, activation, *args, **kwargs)
        self.N_rnn = N_rnn
        self.rnn = nn.GRU(
            input_size=self.F_c[-1] * 2,
            hidden_size=self.F_c[-1],
            num_layers=self.N_rnn,
            batch_first=True,          # PORT: gluon layout='NTC'
        )

    def _rnn_train(self, X, NX, NX_rep, graph_to_rnn, rnn_to_graph, NX_cum):
        X_avg = segment_sum(X, NX_rep, NX.shape[0]) / NX.unsqueeze(1).float()
        X = torch.cat([X_avg, X[NX_cum - 1]], dim=1)          # [m, 2 * F_c[-1]]
        X = X[graph_to_rnn]                                    # [N, K, MAX_ITER, 2*F_c[-1]]
        batch_size, iw_size, length, num_features = X.shape

        out, _ = self.rnn(X.reshape(batch_size * iw_size, length, num_features))
        out = out.reshape(batch_size, iw_size, length, -1)
        return out[rnn_to_graph[0], rnn_to_graph[1], rnn_to_graph[2]]

    def _policy(self, X, A, NX, NX_rep, last_append_mask, graph_to_rnn, rnn_to_graph, NX_cum):
        X = self.embedding_atom(X) + self.embedding_mask(last_append_mask)
        X = self._graph_conv_forward(X, A)
        # PORT: the MXNet version called self.dense(X) twice (same result, but BatchNorm
        # running stats were updated twice per step). Computed once here.
        X_dense = self.dense(X)
        X_mol = self._rnn_train(X_dense, NX, NX_rep, graph_to_rnn, rnn_to_graph, NX_cum)
        return self.policy_h(X_dense, NX, NX_rep, X_mol)


class VanillaMolGen_RNN(MoleculeGenerator_RNN):
    def __init__(self, N_A, N_B, D, F_e, F_h, F_skip, F_c, Fh_policy, activation, N_rnn):
        super(VanillaMolGen_RNN, self).__init__(
            N_A, N_B, D, F_e, F_skip, F_c, Fh_policy, activation, N_rnn, F_h)

    def _build_graph_conv(self, F_h):
        self.F_h = list(F_h) if isinstance(F_h, tuple) else list(F_h)
        self.conv = nn.ModuleList()
        self.bn = nn.ModuleList()          # bn[i-1] belongs to conv[i]; conv[0] has no BN
        for i, (f_in, f_out) in enumerate(zip([self.F_e] + self.F_h[:-1], self.F_h)):
            self.conv.append(GraphConv(f_in, f_out, self.N_B + self.D))
            if i != 0:
                self.bn.append(nn.BatchNorm1d(f_in, eps=1e-5, momentum=0.1))

        self.bn_skip = nn.BatchNorm1d(sum(self.F_h), eps=1e-5, momentum=0.1)
        self.linear_skip = Linear_BN(sum(self.F_h), self.F_skip)

    def _graph_conv_forward(self, X, A):
        X_out = [X]
        for i, conv in enumerate(self.conv):
            X = X_out[-1]
            if i == 0:
                X_out.append(conv(X, A))
            else:
                X_out.append(conv(self.activation(self.bn[i - 1](X)), A))
        X_out = torch.cat(X_out[1:], dim=1)
        return self.activation(self.linear_skip(self.activation(self.bn_skip(X_out))))

## Weight Initialisation

MXNet's `mx.init.Xavier()` default is uniform / factor_type='avg' / magnitude=3, giving bounds
`±sqrt(6/(fan_in+fan_out))` — algebraically identical to `nn.init.xavier_uniform_(w, gain=1)`.
Biases, `policy_0`, and BatchNorm parameters keep their own MXNet initialisers (zeros / ones),
which override the global default in Gluon.

In [11]:
def init_weights_xavier(module):
    for name, p in module.named_parameters():
        if p.dim() < 2:
            continue                                   # biases, BN gamma/beta, policy_0
        if 'bn' in name.split('.')[-1]:
            continue
        nn.init.xavier_uniform_(p, gain=1.0)

    for name, p in module.named_parameters():
        if name.endswith('bias') or ('bias' in name.split('.')[-1]):
            nn.init.zeros_(p)

    for m in module.modules():
        if isinstance(m, nn.BatchNorm1d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
            m.reset_running_stats()

    if hasattr(module, 'policy_0'):
        nn.init.zeros_(module.policy_0)
    return module

## Initialization & Resume Logic

In [12]:
configs = {
    'F_e': F_E,
    'F_h': F_H,
    'F_skip': F_SKIP,
    'F_c': F_C,
    'Fh_policy': FH_POLICY,
    'activation': ACTIVATION,
    'N_rnn': N_RNN,
}

is_continuous = False
global_counter = 0

if not FORCE_RESTART and all(os.path.isfile(f) for f in [CKPT_PARAMS_FILE, CKPT_TRAINER_FILE, LOG_FILE]):
    is_continuous = True
    print("[INFO] Existing checkpoints found. Attempting to resume run.")
    with open(CKPT_CONFIG_FILE, 'r') as f:
        configs = json.load(f)
else:
    print("[INFO] Starting fresh run. Overwriting previous logs/checkpoints.")
    with open(CKPT_CONFIG_FILE, 'w') as f:
        json.dump(configs, f)

model = VanillaMolGen_RNN(
    get_mol_spec().num_atom_types,
    get_mol_spec().num_bond_types,
    D=D_RECEPTIVE,
    **configs
)

if is_continuous:
    state = torch.load(CKPT_PARAMS_FILE, map_location='cpu', weights_only=False)
    model.load_state_dict(state['model'])
    model.to(CTX)
else:
    init_weights_xavier(model)
    model.to(CTX)

# MXNet mx.optimizer.Adam defaults (beta1=0.9, beta2=0.999, eps=1e-8) == torch defaults.
# `wd` was 0 in the original; DECAY above is an LR schedule, not weight decay.
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.0)

if is_continuous:
    tstate = torch.load(CKPT_TRAINER_FILE, map_location='cpu', weights_only=False)
    optimizer.load_state_dict(tstate['optimizer'])
    for g in optimizer.param_groups:
        g['lr'] = tstate.get('lr', LR)

    with open(LOG_FILE, 'r') as f:
        records = [line.strip() for line in f.readlines() if line.strip()]

    if len(records) > 1:
        final_record = records[-1] if records[-1] != 'Training finished' else records[-2]
        try:
            global_counter = int(final_record.split('\t')[0])
            print(f"[INFO] Successfully loaded states. Resuming from Iteration {global_counter}")
        except ValueError:
            print("[WARNING] Could not parse iteration from log. Starting from Iteration 0.")
            global_counter = 0
    else:
        print("[INFO] Log file only contains header. Resuming from Iteration 0.")
        global_counter = 0

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_bn_stats = sum(b.numel() for name, b in model.named_buffers()
                 if name.endswith('running_mean') or name.endswith('running_var'))
print(f"[INFO] Model initialized! Total trainable parameters: {n_trainable:,}")
print(f"[INFO] + BatchNorm running stats: {n_bn_stats:,}  "
      f"=> {n_trainable + n_bn_stats:,} (MXNet reported 6,986,506)")

[INFO] Existing checkpoints found. Attempting to resume run.
[INFO] Successfully loaded states. Resuming from Iteration 406000
[INFO] Model initialized! Total trainable parameters: 6,981,514
[INFO] + BatchNorm running stats: 4,992  => 6,986,506 (MXNet reported 6,986,506)


## Eq. 8 Verification

`context/04` §6.6: *"Verify on a tiny mixed-label batch before any long run: the sign of `l`
must flip on exactly the toxic molecules."* This cell does that, on a real batch.

In [13]:
def verify_eq8(model, batch_cpu):
    """Check that l_weighted == l for non-toxic molecules and == -lambda*l for toxic ones."""
    prev_mode = model.mode
    model.mode = 'likelihood'
    model.eval()
    try:
        with torch.no_grad():
            inputs = batch_to_tensor(batch_cpu)
            l_weighted = model(*inputs)
            l = model.last_l
            tox = model.last_tox

        expected = l * (1.0 - (1.0 + LAMBDA_TOX) * tox)
        max_err = (l_weighted - expected).abs().max().item()

        n_tox = int(tox.sum().item())
        print(f"molecules = {tox.numel()}  toxic = {n_tox}  non-toxic = {tox.numel() - n_tox}")
        print(f"max |l_weighted - l*(1-(1+lambda)*tox)| = {max_err:.3e}")
        if n_tox == 0:
            print("[WARN] No toxic molecules in this batch — sign flip not exercised. "
                  "Re-run on another batch.")
        else:
            flipped = torch.sign(l_weighted[tox > 0]) == -torch.sign(l[tox > 0])
            print(f"sign flipped on toxic molecules: {int(flipped.sum())}/{n_tox}")
            same = torch.sign(l_weighted[tox == 0]) == torch.sign(l[tox == 0])
            print(f"sign preserved on non-toxic:     {int(same.sum())}/{tox.numel() - n_tox}")
        assert max_err < 1e-3, "Eq. 8 weighting does not match the closed form."
        print("[OK] Eq. 8 verified.")
    finally:
        model.mode = prev_mode
        model.train()


verify_eq8(model, batch_cpu)

molecules = 16  toxic = 2  non-toxic = 14
max |l_weighted - l*(1-(1+lambda)*tox)| = 0.000e+00
sign flipped on toxic molecules: 2/2
sign preserved on non-toxic:     14/14
[OK] Eq. 8 verified.


# **The Pretraining Loop**
## Setup

In [14]:
print("-" * 50)
print(f"Starting Pretraining on Device: {CTX}")
print(f"Batch Size: {BATCH_SIZE}, K Paths: {K_PATHS}, Alpha: {P_ALPHA}, Lambda: {LAMBDA_TOX}")
print("-" * 50)

it_train = iter(loader_train)

if not is_continuous:
    t0 = time.time()
    with open(LOG_FILE, 'w') as f:
        # PORT: two extra columns (F12). Columns 0 and 1 are unchanged, so the resume
        # parser above still works on logs written by either version.
        f.write('step\ttime(h)\tloss\tlr\tl_nontoxic\tl_toxic\n')
else:
    with open(LOG_FILE, 'r') as f:
        records = [line.strip() for line in f.readlines() if line.strip()]

    if len(records) > 1:
        final_record = records[-1] if records[-1] != 'Training finished' else records[-2]
        try:
            last_time_hours = float(final_record.split('\t')[1])
            t0 = time.time() - (last_time_hours * 3600)
            print(f"[INFO] Resuming timer from {last_time_hours:.2f} hours.")
        except ValueError:
            print("[WARNING] Could not parse time from log. Starting timer from 0.")
            t0 = time.time()
    else:
        print("Log file only contains header. Starting timer from 0.")
        t0 = time.time()

--------------------------------------------------
Starting Pretraining on Device: cuda:0
Batch Size: 16, K Paths: 5, Alpha: 0.8, Lambda: 1.0
--------------------------------------------------
[INFO] Resuming timer from 7.48 hours.


## Main Training Loop

In [15]:
model.mode = 'loss'
model.train()

while global_counter < TOTAL_ITERATIONS:
    global_counter += 1

    # 1. Fetch next batch safely
    try:
        inputs_cpu = next(it_train)
    except StopIteration:
        it_train = iter(loader_train)
        inputs_cpu = next(it_train)

    if inputs_cpu is None:          # whole batch was filtered out by the OOV shield
        global_counter -= 1
        continue

    # 2. Move to device
    inputs_gpu = batch_to_tensor(inputs_cpu)

    # 3. Forward and backward
    # PORT: MXNet Gluon overwrites grads each backward; PyTorch accumulates. zero_grad is required.
    optimizer.zero_grad(set_to_none=True)
    loss = model(*inputs_gpu)       # -mean(l_weighted)
    loss.backward()

    # 4. Gradient clip + optimizer step
    # PORT: MXNet's optimizer `clip_gradient=3.0` clips element-wise -> clip_grad_value_.
    #       `trainer.step(batch_size=1)` applied no rescaling, so nothing else is needed.
    torch.nn.utils.clip_grad_value_(model.parameters(), CLIP_GRAD)
    optimizer.step()

    # 5. Learning-rate decay
    if global_counter % DECAY_STEP == 0:
        for g in optimizer.param_groups:
            g['lr'] = g['lr'] * (1.0 - DECAY)

    # 6. Periodic logging and checkpointing
    if global_counter % SUMMARY_STEP == 0:
        loss_val = loss.item()
        lr_now = optimizer.param_groups[0]['lr']
        elapsed_hours = float(time.time() - t0) / 3600.0

        # F12: split the two branches of Eq. 8 so suppression can be seen separately.
        _l, _tox = model.last_l, model.last_tox
        l_non = _l[_tox == 0].mean().item() if (_tox == 0).any() else float('nan')
        l_tox = _l[_tox > 0].mean().item() if (_tox > 0).any() else float('nan')

        torch.save({'model': model.state_dict()}, CKPT_PARAMS_FILE)
        torch.save({'optimizer': optimizer.state_dict(),
                    'lr': lr_now,
                    'global_counter': global_counter}, CKPT_TRAINER_FILE)

        log_str = (f"{global_counter}\t{elapsed_hours:.4f}\t{loss_val:.4f}\t{lr_now}"
                   f"\t{l_non:.4f}\t{l_tox:.4f}\n")
        with open(LOG_FILE, 'a') as f:
            f.write(log_str)
            f.flush()

        print(f"Iter: {global_counter:06d} | Loss: {loss_val:.4f} | "
              f"l_nontox: {l_non:.3f} | l_tox: {l_tox:.3f} | "
              f"LR: {lr_now:.6e} | Time: {elapsed_hours:.2f}h")

        del loss, inputs_cpu, inputs_gpu
        gc.collect()

Iter: 406500 | Loss: -22.7737 | l_nontox: -48.888 | l_tox: -237.760 | LR: 1.712812e-05 | Time: 7.49h
Iter: 407000 | Loss: 27.4384 | l_nontox: -47.714 | l_tox: -114.488 | LR: 1.704265e-05 | Time: 7.49h
Iter: 407500 | Loss: -31.6480 | l_nontox: -59.543 | l_tox: -232.268 | LR: 1.695761e-05 | Time: 7.50h
Iter: 408000 | Loss: 51.8046 | l_nontox: -66.168 | l_tox: -163.654 | LR: 1.687299e-05 | Time: 7.51h
Iter: 408500 | Loss: -4.5996 | l_nontox: -135.573 | l_tox: -312.980 | LR: 1.678879e-05 | Time: 7.52h
Iter: 409000 | Loss: -30.3756 | l_nontox: -39.822 | l_tox: -334.566 | LR: 1.670501e-05 | Time: 7.52h
Iter: 409500 | Loss: -40.0576 | l_nontox: -53.837 | l_tox: -321.740 | LR: 1.662166e-05 | Time: 7.53h
Iter: 410000 | Loss: 23.4600 | l_nontox: -78.553 | l_tox: -362.191 | LR: 1.653871e-05 | Time: 7.54h
Iter: 410500 | Loss: -10.2104 | l_nontox: -45.137 | l_tox: -102.456 | LR: 1.645619e-05 | Time: 7.55h
Iter: 411000 | Loss: -3.9242 | l_nontox: -30.931 | l_tox: -247.913 | LR: 1.637407e-05 | Time: 

## Final Save & Shutdown

In [16]:
print("Target iterations reached. Saving final checkpoint...")
torch.save({'model': model.state_dict()}, CKPT_PARAMS_FILE)
torch.save({'optimizer': optimizer.state_dict(),
            'lr': optimizer.param_groups[0]['lr'],
            'global_counter': global_counter}, CKPT_TRAINER_FILE)

with open(LOG_FILE, 'a') as f:
    f.write('Training finished\n')

print("Pretraining Complete!")

Target iterations reached. Saving final checkpoint...
Pretraining Complete!
